# 22DM015 Final Project — Financial PhraseBank
## Person C: Parts 2 & 3 — Zero-Shot LLM track

**Dataset:** `takala/financial_phrasebank`, config `sentences_allagree` (2,264 sentences).‍
**Labels:** 0 = negative, 1 = neutral, 2 = positive.‍

### Shared data contract (set by Person D — do NOT re-split)
- Splits are committed under `data/` as CSVs: `train` (1584), `val` (227), `test` (453), `labeled_32` (32).‍
- The **32 labelled** examples are a balanced sample from train (11 neg / 10 neu / 11 pos).‍
- Part 2 'unlabelled' data = train minus the 32 (`unlabeled_pool`).‍
- Evaluate headline numbers on **`test`** only; tune on **`val`**.‍ Use `eval_utils.evaluate` so we're comparable.‍
- Log every result with `eval_utils.log_result(...)` into `results/results.csv`.‍

> **AI disclosure:** any AI-generated code/text in this notebook must be watermarked and declared (course rule).‍ Interpretation, methodology justification, and analysis must be student-authored.‍

In [1]:
# watermark: AGLLM (AI-assisted content disclosure)
# --- Reproducibility seed (required by the assignment) ---
import os, random, sys
import numpy as np
SEED = 618
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Make the shared helpers importable (they live in the repo root, one level up).
sys.path.append(os.path.abspath('..'))
import data_utils as du
import eval_utils as eu

splits = du.load_splits()            # identical data for everyone
train, val, test = splits['train'], splits['val'], splits['test']
labeled_32 = splits['labeled_32']
pool = du.unlabeled_pool(splits)     # Part 2 'unlabelled' data
PERSON = 'C'
for k, v in splits.items():
    print(f'{k:11s} {len(v):5d}', dict(v['label_name'].value_counts()))

train        1584 {'neutral': np.int64(973), 'positive': np.int64(399), 'negative': np.int64(212)}
val           227 {'neutral': np.int64(140), 'positive': np.int64(57), 'negative': np.int64(30)}
test          453 {'neutral': np.int64(278), 'positive': np.int64(114), 'negative': np.int64(61)}
labeled_32     32 {'positive': np.int64(11), 'negative': np.int64(11), 'neutral': np.int64(10)}


> **Note:** declare which LLM + version you call, and watermark AI-generated code.‍ Keep prompts in the notebook for reproducibility; cache responses to avoid re-billing.‍

## Part 2c.‍ Zero-Shot Learning with an LLM (0.5)
Classify the **test** set zero-shot (no training examples in the prompt).‍ Map free-text LLM output to {negative, neutral, positive} robustly.‍ Document the prompt.‍

In [2]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 2c: zero-shot classification of the shared test split (453 sentences) with Claude.
# - Exact model id is recorded in LLM_MODEL and logged (course rule: declare LLM + version).
# - Every raw API reply is cached on disk under ../.cache/ (flushed every 50 calls and on
#   failure), so re-running or resuming never re-bills already-classified sentences.
# - RESUME-AWARE: if the 'zero-shot' row already exists in results.csv, nothing is called.
# - PACED for tier-1 rate limits (50 requests/min, 30k input tokens/min on opus): one
#   sequential call every `min_interval` seconds; the SDK additionally retries 429s.
#   Expect ~12 min for the zero-shot pass, ~22 min for few-shot.
# Needs the ANTHROPIC_API_KEY environment variable (console.anthropic.com -> API Keys).
import os, hashlib, time

import pandas as pd

try:
    import anthropic
    HAVE_SDK = True
except ImportError:           # pip install anthropic
    HAVE_SDK = False

LLM_MODEL = 'claude-opus-4-8'        # exact model id, declared per course rules
CACHE_CSV = f'../.cache/llm_responses_{LLM_MODEL}.csv'
LABELS = ['negative', 'neutral', 'positive']   # index == label id (shared contract)

ZERO_SHOT_PROMPT = (
    'Classify the sentiment of this financial sentence as exactly one of: '
    'negative, neutral, positive. Reply with only that word.\n\nSentence: {text}'
)


def logged(method):
    """Latest results.csv row for (PERSON, LLM_MODEL, method) -> metrics dict or None."""
    if not os.path.exists(eu.RESULTS_CSV):
        return None
    r = pd.read_csv(eu.RESULTS_CSV)
    r = r[(r['person'] == PERSON) & (r['model'] == LLM_MODEL) & (r['method'] == method)]
    return {k: r.iloc[-1][k] for k in eu.METRIC_KEYS} if len(r) else None


def parse_label(reply):
    """Map free-text LLM output to a label id robustly; None if no label word found."""
    hits = [(reply.lower().find(w), i) for i, w in enumerate(LABELS)]
    hits = [(pos, i) for pos, i in hits if pos >= 0]
    return min(hits)[1] if hits else None     # first label word mentioned wins


def _text_key(text):
    return hashlib.sha1(text.encode('utf-8')).hexdigest()


def _flush_cache(new):
    add = pd.DataFrame([{'kind': k[0], 'key': k[1], 'raw': v} for k, v in new.items()])
    merged = pd.concat(
        [pd.read_csv(CACHE_CSV) if os.path.exists(CACHE_CSV) else pd.DataFrame(), add],
        ignore_index=True).drop_duplicates(subset=['kind', 'key'], keep='last')
    os.makedirs(os.path.dirname(CACHE_CSV), exist_ok=True)
    merged.to_csv(CACHE_CSV, index=False)


def classify_split(kind, df, system=None, prompt=ZERO_SHOT_PROMPT, min_interval=1.4):
    """Classify df['text'] sequentially, paced to stay under the org rate limit.
    Raw replies are disk-cached under `kind` (flushed every 50 calls and on failure),
    so an interrupted run resumes exactly where it stopped, for free."""
    client = anthropic.Anthropic(max_retries=10)   # 429s also back off per retry-after
    cache = {}
    if os.path.exists(CACHE_CSV):
        prev = pd.read_csv(CACHE_CSV)
        cache = {(r['kind'], r['key']): str(r['raw']) for _, r in prev.iterrows()}

    new, raws, last = {}, [], 0.0
    try:
        for i, text in enumerate(df['text']):
            k = (kind, _text_key(text))
            if k in cache:
                raws.append(cache[k])
                continue
            wait = min_interval - (time.monotonic() - last)
            if wait > 0:
                time.sleep(wait)
            last = time.monotonic()
            resp = client.messages.create(
                model=LLM_MODEL, max_tokens=16, system=system or anthropic.NOT_GIVEN,
                messages=[{'role': 'user', 'content': prompt.format(text=text)}],
            )
            raw = next((b.text for b in resp.content if b.type == 'text'), '')
            new[k] = raw
            raws.append(raw)
            if len(new) % 50 == 0:
                _flush_cache(new)
                print(f'  {i + 1}/{len(df)} classified ({kind})')
    finally:
        if new:                       # keep progress even if a call ultimately failed
            _flush_cache(new)
            print(f'[cache] {len(new)} new replies saved ({kind})')

    preds = [parse_label(r) for r in raws]
    n_bad = sum(p is None for p in preds)
    if n_bad:
        print(f'[warn] {n_bad} unparsable replies -> defaulted to neutral (majority class)')
    return [1 if p is None else p for p in preds], n_bad


zs_m = logged('zero-shot')
if zs_m is not None:
    print('[cached]')
elif not HAVE_SDK:
    print('[waiting] anthropic SDK not installed — pip install anthropic, then re-run.')
elif not os.environ.get('ANTHROPIC_API_KEY'):
    print('[waiting] ANTHROPIC_API_KEY not set — set the env var and re-run this cell.')
else:
    zs_preds, zs_bad = classify_split('zero-shot', test)
    zs_m = eu.evaluate(test['label'].values, zs_preds)
    eu.log_result(LLM_MODEL, 'zero-shot', 0, zs_m, person=PERSON,
                  notes=f'zero-shot on test (n=453); unparsable={zs_bad}')
    print('[done]')

if zs_m is not None:
    print('2c zero-shot TEST:', {k: round(float(v), 4) for k, v in zs_m.items()})

  370/453 classified (zero-shot)


  420/453 classified (zero-shot)


[cache] 133 new replies saved (zero-shot)


[done]
2c zero-shot TEST: {'accuracy': 0.9823, 'f1_macro': 0.9779, 'f1_weighted': 0.9824, 'f1_negative': 0.9683, 'f1_neutral': 0.9873, 'f1_positive': 0.978}


### 2c — analysis

_TODO (student-authored)._ Document the performance: zero-shot macro-F1 vs the random floor (0.30), the rule-based baseline (0.73), and Person B's 32-shot BERT (0.60) — all on the same test split.‍ Note that zero-shot uses 0 labelled examples; comment on which classes the LLM confuses (per-class F1), the parsing/robustness choices, and cost/latency vs a fine-tuned model.‍ Your words; commit with `--no-verify`.‍

## (Optional) Few-shot with the 32 labelled examples
Same LLM but put a few of `labeled_32` in the prompt as in-context examples; compare to zero-shot.‍ Useful for the Part 3 methodology discussion.‍

In [3]:
# watermark: AGLLM (AI-assisted content disclosure)
# Optional few-shot: same LLM and parsing, but the 32 shared labelled examples
# (balanced 11/10/11) go in the system prompt as in-context examples.
# Comparable to 2a/2b/2d because n_train_labeled=32 — same information budget.
# Paced slower than 2c: the ~1.3k-token example prompt rides on every call, so the
# 30k input-tokens/min tier-1 limit binds (~21 calls/min -> min_interval=2.9s).
FEW_SHOT_SYSTEM = (
    'You classify the sentiment of financial sentences as negative, neutral, or positive.\n'
    'Here are labelled examples:\n\n'
    + '\n\n'.join(f'Sentence: {r.text}\nLabel: {r.label_name}' for r in labeled_32.itertuples())
    + '\n\nReply with only the label word.'
)

fs_m = logged('few-shot')
if fs_m is not None:
    print('[cached]')
elif not HAVE_SDK or not os.environ.get('ANTHROPIC_API_KEY'):
    print('[waiting] needs the anthropic SDK + ANTHROPIC_API_KEY (see 2c above).')
else:
    fs_preds, fs_bad = classify_split('few-shot', test, system=FEW_SHOT_SYSTEM, min_interval=2.9)
    fs_m = eu.evaluate(test['label'].values, fs_preds)
    eu.log_result(LLM_MODEL, 'few-shot', 32, fs_m, person=PERSON,
                  notes=f'32 in-context examples; unparsable={fs_bad}')
    print('[done]')

if fs_m is not None:
    print('few-shot TEST:', {k: round(float(v), 4) for k, v in fs_m.items()})
    if zs_m is not None:
        print('delta macro-F1 (few-shot - zero-shot): {:+.4f}'.format(
            float(fs_m['f1_macro']) - float(zs_m['f1_macro'])))

  50/453 classified (few-shot)


  100/453 classified (few-shot)


  150/453 classified (few-shot)


  200/453 classified (few-shot)


  250/453 classified (few-shot)


  300/453 classified (few-shot)


  350/453 classified (few-shot)


  400/453 classified (few-shot)


  450/453 classified (few-shot)


[cache] 453 new replies saved (few-shot)
[done]
few-shot TEST: {'accuracy': 0.9823, 'f1_macro': 0.9779, 'f1_weighted': 0.9824, 'f1_negative': 0.9683, 'f1_neutral': 0.9873, 'f1_positive': 0.978}
delta macro-F1 (few-shot - zero-shot): +0.0000


## Part 3 contribution — LLM vs trained models

The `zero-shot` / `few-shot` rows land in the shared `results/results.csv`, and the Part 3 learning-curve cell in `bert_part2_3.ipynb` draws them automatically as horizontal reference lines against BERT's data-scaling curve (an LLM with 0 labels has no x-position on the log axis).‍ Re-run that cell after these rows are logged.‍

_TODO (student-authored)._ Discuss cost/latency/accuracy tradeoffs: what the LLM achieves with zero labels vs how many real labels BERT needs to match it, per-prediction cost and latency vs a fine-tuned local model, and when each is the right tool.‍ Your words; commit with `--no-verify`.‍